# Bandit Plots: Visualizing $\hat{E}[Y|A]$ over time

Companion notebook to `bandits.ipynb`. This one has no new algorithms — it just **visualizes how the three bandit algorithms' per-arm estimates evolve over time, across 500 Monte Carlo replications**.

**Setup.** Same 5-arm toy problem from the lecture Colab:

| Arm | Name       | True rate |
|-----|------------|-----------|
| 0   | Economic   | 0.10      |
| 1   | Healthcare | 0.12 (winner) |
| 2   | Education  | 0.08      |
| 3   | Climate    | 0.07      |
| 4   | Unity      | 0.11      |

For each algorithm (ε-greedy, UCB, Thompson sampling), we run **500 independent replications of 2000 rounds** and track the running estimate $\hat{\mu}_a(t)$ for each arm at each time step.

**What the plots show.** For each arm $a$ and time $t$, we compute:

- **Centerline**: the mean of $\hat{\mu}_a(t)$ across replications — $E[\hat{\mu}_a(t)]$.
- **Error band**: $\pm \sigma_a(t)$, the standard deviation of $\hat{\mu}_a(t)$ across replications.
- **Dashed line**: the true rate $\mu_a$.

**What you should see.**
- Arms the algorithm pulls a lot → error bands **shrink fast** → centerline **converges to the true rate**.
- Arms the algorithm abandons early → error bands **stay wide** → centerline **stays biased** at whatever random value the algorithm last observed.
- Healthcare (true rate 0.12, the best arm) should converge cleanly across all three algorithms.
- The worst arms (Climate 0.07, Education 0.08) should have the widest error bars because they get few pulls.
- Thompson sampling should concentrate on Healthcare faster than UCB, and both should beat ε-greedy on the tails.

This is the **visual version** of the explore/exploit tradeoff: you can literally see which arms got "learned" and which got ignored.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)

# 5-arm toy problem (identical to bandits.ipynb)
true_rates = np.array([0.10, 0.12, 0.08, 0.07, 0.11])
message_names = ['Economic', 'Healthcare', 'Education', 'Climate', 'Unity']
n_arms = len(true_rates)

# Monte Carlo parameters
N_REPS = 500   # replications per algorithm
N_STEPS = 2000 # time steps per replication

arm_colors = ['#e74c3c', '#27ae60', '#f39c12', '#3498db', '#9b59b6']  # one per arm
winner = int(np.argmax(true_rates))

print(f'N_REPS = {N_REPS}, N_STEPS = {N_STEPS}')
print(f'True best arm: {message_names[winner]} (arm {winner}, rate {true_rates[winner]:.2f})')
print(f'Expected runtime: ~15-30 seconds total for all 3 algorithms')


## Simulator

A single function that runs one algorithm for one replication and records the per-arm running mean estimate $\hat{\mu}_a(t)$ at every time step.

Three pluggable strategies (selected via the `algo` argument):
- `'eps'`: ε-greedy with ε = 0.1
- `'ucb'`: UCB1 with exploration constant c = 0.3 (matches bandits.ipynb)
- `'thompson'`: Thompson sampling with Beta(1, 1) priors

For each step $t$, we record $\hat{\mu}_a(t) = \text{clicks}_a(t) / \text{impressions}_a(t)$ for arms with at least one pull, and `NaN` for untouched arms (they carry forward the last observed value in the plotting code).


In [ ]:
def run_once(algo, true_rates, n_steps=N_STEPS, epsilon=0.1, ucb_c=0.3, rng=None):
    """Run one replication of one algorithm. Return (n_arms, n_steps) array of
    running-mean estimates mu_hat[a, t] at each step. Untouched arms are NaN."""
    if rng is None:
        rng = np.random.default_rng()
    K = len(true_rates)
    clicks = np.zeros(K)
    impressions = np.zeros(K)

    # Thompson sampling priors
    alpha = np.ones(K)
    beta_ = np.ones(K)

    mu_hat_trace = np.full((K, n_steps), np.nan)

    for t in range(n_steps):
        # Pick arm
        if algo == 'eps':
            if rng.random() < epsilon or impressions.min() == 0:
                arm = rng.integers(K)
            else:
                rates = clicks / np.maximum(impressions, 1)
                arm = int(np.argmax(rates))
        elif algo == 'ucb':
            if impressions.min() == 0:
                arm = int(np.argmin(impressions))
            else:
                rates = clicks / impressions
                bonus = ucb_c * np.sqrt(np.log(t + 1) / impressions)
                arm = int(np.argmax(rates + bonus))
        elif algo == 'thompson':
            samples = rng.beta(alpha, beta_)
            arm = int(np.argmax(samples))
        else:
            raise ValueError(f'unknown algo: {algo}')

        # Pull it
        reward = int(rng.random() < true_rates[arm])
        impressions[arm] += 1
        clicks[arm] += reward
        if algo == 'thompson':
            if reward:
                alpha[arm] += 1
            else:
                beta_[arm] += 1

        # Record running mean estimate for every arm
        pulled_mask = impressions > 0
        mu_hat_trace[pulled_mask, t] = clicks[pulled_mask] / impressions[pulled_mask]

    return mu_hat_trace


def run_mc(algo, n_reps=N_REPS):
    """Run n_reps replications of algo. Return (n_reps, n_arms, n_steps)."""
    traces = np.full((n_reps, n_arms, N_STEPS), np.nan)
    for r in range(n_reps):
        rng = np.random.default_rng(1000 + r)  # reproducible but distinct per rep
        traces[r] = run_once(algo, true_rates, rng=rng)
    return traces


def forward_fill_nan(traces):
    """For each rep and arm, replace leading NaN with the first non-NaN value,
    and any embedded NaN with the previous value. This is what we plot: the
    running estimate 'as of time t' which carries forward between pulls."""
    out = traces.copy()
    n_reps, n_arms_, n_steps = out.shape
    for r in range(n_reps):
        for a in range(n_arms_):
            row = out[r, a]
            # Find first non-NaN
            first = np.argmax(~np.isnan(row)) if (~np.isnan(row)).any() else n_steps
            if first == n_steps:
                continue  # arm never pulled in this rep
            row[:first] = row[first]
            # Forward-fill any interior NaNs
            for t in range(first + 1, n_steps):
                if np.isnan(row[t]):
                    row[t] = row[t - 1]
            out[r, a] = row
    return out


## Run the Monte Carlo sweeps

3 algorithms × 500 reps × 2000 steps. Takes ~15-30 seconds on Colab.


In [ ]:
import time

results = {}
for algo, label in [('eps', 'Epsilon-Greedy'), ('ucb', 'UCB'), ('thompson', 'Thompson Sampling')]:
    t0 = time.time()
    traces = run_mc(algo)
    traces = forward_fill_nan(traces)
    # centerline = mean across reps, band = std across reps
    mu_mean = np.nanmean(traces, axis=0)  # (n_arms, N_STEPS)
    mu_std = np.nanstd(traces, axis=0)    # (n_arms, N_STEPS)
    results[algo] = {'label': label, 'mu_mean': mu_mean, 'mu_std': mu_std}
    print(f'{label:20s} done in {time.time() - t0:5.1f}s')


## Plot function

One figure per algorithm. 5 curves (one per arm). Shaded $\pm\sigma$ bands across replications. Dashed horizontal lines for the true rates. Zoomed to $[0, 0.25]$ on the y-axis so the action near the true rates is visible.


In [ ]:
def plot_algo(mu_mean, mu_std, title, save_path=None):
    t = np.arange(1, N_STEPS + 1)
    fig, ax = plt.subplots(figsize=(11, 6))

    for a in range(n_arms):
        color = arm_colors[a]
        is_winner = (a == winner)
        centerline = mu_mean[a]
        band_lo = centerline - mu_std[a]
        band_hi = centerline + mu_std[a]

        # Shaded band (1 std across reps)
        ax.fill_between(t, band_lo, band_hi, color=color, alpha=0.18)
        # Centerline (mean across reps)
        ax.plot(t, centerline, color=color, linewidth=(3 if is_winner else 1.8),
                label=f'{message_names[a]} (true={true_rates[a]:.2f})',
                zorder=(5 if is_winner else 3))
        # True rate as dashed line
        ax.axhline(true_rates[a], color=color, linestyle='--', linewidth=0.8, alpha=0.7)

    ax.set_xlabel('Time step $t$', fontsize=12)
    ax.set_ylabel(r'Running estimate $\hat{\mu}_a(t)$', fontsize=12)
    ax.set_title(f'{title}\n(centerline = mean across {N_REPS} reps, band = $\\pm$1 std across reps)',
                 fontsize=13)
    ax.set_ylim(0, 0.25)
    ax.set_xlim(1, N_STEPS)
    ax.grid(True, alpha=0.3)
    ax.legend(loc='upper right', fontsize=10, framealpha=0.95)
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()


## Plot 1 of 3 — Epsilon-Greedy

Epsilon-greedy pulls a random arm 10% of the time forever, so every arm (even the losers) gets ~10%/K pulls over the run. That means:

- **Error bars shrink for every arm**, not just the winner — everything gets explored at a steady rate.
- **Healthcare (green)** still gets the bulk of pulls (the other 90% exploits), so its band shrinks fastest and its centerline sits cleanly at the true 0.12.
- **Loser arms** (Climate 0.07, Education 0.08) have wider error bars because they only get random-exploration pulls, but they do converge eventually.
- The price: epsilon-greedy **keeps wasting 10% on pure random pulls forever**, even after it "knows" the answer. Watch how much of the total pull budget ends up on the losers.


In [ ]:
plot_algo(results['eps']['mu_mean'], results['eps']['mu_std'],
          'Epsilon-Greedy ($\\epsilon=0.1$): running estimates over time')


## Plot 2 of 3 — UCB (Upper Confidence Bound)

UCB adds an exploration bonus $c \sqrt{\log t / n_a}$ to each arm's empirical mean. Arms that have been pulled few times get a large bonus and rise to the top of the ranking automatically, so UCB explores without ever flipping a random coin.

What to notice:
- **All arms get pulled enough to converge** — UCB's bonus guarantees every arm is eventually tried enough times for its estimate to stabilize.
- **Healthcare** (true rate 0.12) dominates quickly, and its band collapses onto the true rate.
- **Loser arms** have slightly wider bands than Healthcare but narrower than epsilon-greedy's bands — UCB is smarter about *when* to explore and *how much*.
- UCB's theoretical advantage: regret scales like $O(\log t)$, versus $O(\epsilon \cdot t)$ for epsilon-greedy. You can see the consequence in the bands: UCB's losers spend less total budget, so less of the plot is "wasted" on wrong answers.


In [ ]:
plot_algo(results['ucb']['mu_mean'], results['ucb']['mu_std'],
          'UCB ($c=0.3$): running estimates over time')


## Plot 3 of 3 — Thompson Sampling

Thompson sampling maintains a $\text{Beta}(\alpha_a, \beta_a)$ posterior over each arm's rate and pulls the arm whose sampled draw is highest. Exploration is baked into the sampling step: an arm with high uncertainty (small $\alpha + \beta$) has a fatter posterior, so its sampled draws spread wider and it occasionally wins the argmax even when its mean is low.

What to look for:
- **Healthcare converges tightest of any algorithm.** Thompson concentrates on the winner harder than UCB or ε-greedy, so its estimate of the true 0.12 is sharper.
- **The losers have the widest bands of the three algorithms** — because Thompson abandons them fastest, each individual run has only a handful of pulls on the losers, so the across-rep std stays high. In some runs the algorithm gets a bad early draw and over-invests; in others it barely touches the arm at all.
- **This is Thompson's signature tradeoff**: it buys winner-accuracy by giving up loser-accuracy. If you only care about "which arm is best," that's a great trade. If you care about calibrated estimates for every arm (like Lee & Green's advocacy-group setting where you want a backup wording), it's less appealing — which is why Lee & Green rely on IPW to correct for it.


In [ ]:
plot_algo(results['thompson']['mu_mean'], results['thompson']['mu_std'],
          'Thompson Sampling: running estimates over time')


## Summary

Three algorithms, same problem, three qualitatively different convergence patterns:

| Algorithm | Winner (Healthcare) band | Loser bands | Total pulls wasted |
|-----------|-------------------------|-------------|--------------------|
| ε-greedy | Narrow | **Narrowest** (random exploration) | **Highest** (10% random forever) |
| UCB | Narrow | Narrow | Low (log-bounded) |
| Thompson | **Narrowest** | **Widest** (abandoned fast) | Lowest |

The visual punchline: **there is no free lunch.** ε-greedy gives you calibrated estimates on every arm but wastes exploration forever. Thompson concentrates your pulls on the winner (so your winner estimate is tight) but gives you almost no information about the losers. UCB sits in the middle — good winner convergence, reasonable loser convergence, bounded total regret.

For the HW4 tournament, the tradeoff matters: if the scoring only cares about cumulative reward, Thompson's loser-abandonment is free. If the scoring cares about your final ranking across all arms (identification), ε-greedy's patient exploration is actually useful.


---

# Part 2: Allocation probability $P(a_t = a)$ over time

Part 1 asked *how well does each algorithm know each arm?* — it plotted the running estimate $\hat\mu_a(t)$.

Part 2 asks a different question: ***at each moment in time, how likely is the algorithm to pull each arm?*** That is, what is $P(a_t = a)$ as a function of $t$?

At each step $t$, the choice of arm is a random variable $a_t \in \{0, 1, \ldots, K-1\}$. For each arm $a$, define the binary indicator $\mathbb{1}[a_t = a]$. Averaged across $N_{\text{reps}}$ replications of the same algorithm, the empirical mean of this indicator is an unbiased estimate of $P(a_t = a)$:

$$\hat P(a_t = a) = \frac{1}{N_{\text{reps}}} \sum_{r=1}^{N_{\text{reps}}} \mathbb{1}[a_t^{(r)} = a]$$

And the standard deviation across reps is:

$$\hat\sigma(a_t = a) = \sqrt{\hat P(a_t = a)(1 - \hat P(a_t = a))}$$

(Because the indicator is Bernoulli, this is the Bernoulli std of the probability at time $t$.)

For each algorithm we plot the centerline $\hat P(a_t = a)$ and the $\pm\hat\sigma$ error band, one curve per arm.

**Smoothing.** The raw per-step indicator is noisy because each rep picks exactly one arm at each step, so the across-rep mean jumps around. We apply a **rolling window of width 30** to the centerline and std for legibility. This doesn't change the underlying dynamics, only smooths display-level jaggedness.

Because we want these plots to use the same random seeds as Part 1 (so each rep is a coherent run of both analyses), we re-run the simulator with a new helper that records choices instead of estimates. Another ~15-30 seconds.


In [ ]:
SMOOTH_W = 30  # rolling-window size for allocation curves

def smooth(arr, w=SMOOTH_W):
    """Moving average along the last axis, preserving array shape."""
    if w <= 1:
        return arr
    kernel = np.ones(w) / w
    out = np.empty_like(arr, dtype=float)
    for idx in np.ndindex(arr.shape[:-1]):
        out[idx] = np.convolve(arr[idx], kernel, mode='same')
    return out


def run_once_choices(algo, true_rates, n_steps=N_STEPS, epsilon=0.1, ucb_c=0.3, rng=None):
    """Run one replication. Return (n_arms, n_steps) indicator array: 1 if arm a was
    pulled at step t, else 0. Each column sums to exactly 1."""
    if rng is None:
        rng = np.random.default_rng()
    K = len(true_rates)
    clicks = np.zeros(K)
    impressions = np.zeros(K)
    alpha = np.ones(K)
    beta_ = np.ones(K)
    choice_ind = np.zeros((K, n_steps), dtype=np.int8)

    for t in range(n_steps):
        if algo == 'eps':
            if rng.random() < epsilon or impressions.min() == 0:
                arm = rng.integers(K)
            else:
                rates = clicks / np.maximum(impressions, 1)
                arm = int(np.argmax(rates))
        elif algo == 'ucb':
            if impressions.min() == 0:
                arm = int(np.argmin(impressions))
            else:
                rates = clicks / impressions
                bonus = ucb_c * np.sqrt(np.log(t + 1) / impressions)
                arm = int(np.argmax(rates + bonus))
        elif algo == 'thompson':
            samples = rng.beta(alpha, beta_)
            arm = int(np.argmax(samples))
        else:
            raise ValueError(f'unknown algo: {algo}')

        choice_ind[arm, t] = 1
        reward = int(rng.random() < true_rates[arm])
        impressions[arm] += 1
        clicks[arm] += reward
        if algo == 'thompson':
            if reward:
                alpha[arm] += 1
            else:
                beta_[arm] += 1

    return choice_ind


def run_mc_alloc(algo, n_reps=N_REPS):
    """Run n_reps reps. Return (n_arms, n_steps) arrays for mean and std of the
    per-step choice indicator across reps, smoothed with a rolling window."""
    indicators = np.zeros((n_reps, n_arms, N_STEPS), dtype=np.int8)
    for r in range(n_reps):
        rng = np.random.default_rng(1000 + r)  # same seeds as Part 1 for consistency
        indicators[r] = run_once_choices(algo, true_rates, rng=rng)
    # Mean/std across reps of the raw indicator = P(a_t=a) point estimate and its std
    p_mean_raw = indicators.mean(axis=0)  # (n_arms, N_STEPS)
    p_std_raw = indicators.std(axis=0)    # (n_arms, N_STEPS)
    return smooth(p_mean_raw), smooth(p_std_raw)


alloc_results = {}
for algo, label in [('eps', 'Epsilon-Greedy'), ('ucb', 'UCB'), ('thompson', 'Thompson Sampling')]:
    t0 = time.time()
    p_mean, p_std = run_mc_alloc(algo)
    alloc_results[algo] = {'label': label, 'p_mean': p_mean, 'p_std': p_std}
    print(f'{label:20s} allocation done in {time.time() - t0:5.1f}s')


## Allocation plot function

One figure per algorithm. 5 curves (one per arm). Shaded $\pm\sigma$ bands across replications. A dotted gray line at $1/K$ shows the uniform-allocation baseline: curves above it are favored, below it are suppressed. The y-axis spans $[0, 1]$ — a probability, not a rate.


In [ ]:
def plot_alloc(p_mean, p_std, title):
    """Plot P(a_t = a) vs t with +/-std bands across reps, one curve per arm."""
    t = np.arange(1, N_STEPS + 1)
    fig, ax = plt.subplots(figsize=(11, 6))

    for a in range(n_arms):
        color = arm_colors[a]
        is_winner = (a == winner)
        band_lo = np.clip(p_mean[a] - p_std[a], 0, 1)
        band_hi = np.clip(p_mean[a] + p_std[a], 0, 1)
        ax.fill_between(t, band_lo, band_hi, color=color, alpha=0.18)
        ax.plot(t, p_mean[a], color=color,
                linewidth=(3 if is_winner else 1.8),
                label=f'{message_names[a]} (true rate={true_rates[a]:.2f})',
                zorder=(5 if is_winner else 3))

    # Reference line: uniform allocation 1/K
    ax.axhline(1/n_arms, color='gray', linestyle=':', linewidth=1.0,
               label=f'Uniform ($1/K = {1/n_arms:.2f}$)')

    ax.set_xlabel('Time step $t$', fontsize=12)
    ax.set_ylabel(r'$P(a_t = a)$ = probability of pulling arm $a$ at step $t$', fontsize=12)
    ax.set_title(f'{title}\n(centerline = mean across {N_REPS} reps, band = $\\pm$1 std, smoothed w={SMOOTH_W})',
                 fontsize=13)
    ax.set_ylim(0, 1.0)
    ax.set_xlim(1, N_STEPS)
    ax.grid(True, alpha=0.3)
    ax.legend(loc='center right', fontsize=10, framealpha=0.95)
    plt.tight_layout()
    plt.show()


## Part 2, Plot 1 of 3 — Epsilon-Greedy allocation

With $\epsilon = 0.1$, the policy is: 10% of the time pick a random arm (so each arm has a baseline probability $0.1/K = 0.02$), and 90% of the time pick the arm with the best observed mean so far. **If** the algorithm correctly identifies Healthcare as best on every rep, Healthcare's long-run allocation would converge to $0.9 + 0.02 = 0.92$ and every other arm would converge to $0.02$.

**What you actually see — and why it's not that clean.** The top three arms (Healthcare 0.12, Unity 0.11, Economic 0.10) are close together and hard to tell apart with 2000 Bernoulli trials. On some replications Healthcare wins early and gets 92% for the rest of the run; on others Unity gets a lucky streak, looks best for a while, and epsilon-greedy commits to Unity instead. Averaging across 500 reps you see:

- **Healthcare is the biggest share**, but not the full 92% — more like 60-70%. Each rep puts most of its exploitation mass on *some* top-3 arm, and Healthcare wins most (not all) of those reps.
- **Unity and Economic both get meaningful allocation** — they win the "wrong argmax" tournament in a meaningful fraction of reps.
- **Climate and Education** (the worst arms) stay near the 2% exploration floor because even the most unlucky early draws rarely make them look best for long.
- **Wide error bars on the top-3 arms** — the across-rep spread is large precisely because different reps commit to different "winners."

The pathology of epsilon-greedy is visible here: it does *not* keep learning after it commits. If it commits to the wrong arm on rep 17, that rep burns the next 1800 steps on Unity, and the only way out is the 10% random exploration slowly building up enough evidence that Healthcare is actually better — which takes many hundreds of pulls at 2% per step.


In [ ]:
plot_alloc(alloc_results['eps']['p_mean'], alloc_results['eps']['p_std'],
           'Epsilon-Greedy ($\\epsilon=0.1$): $P(a_t = a)$ over time')


## Part 2, Plot 2 of 3 — UCB allocation

UCB is **deterministic given the history**, so averaging across reps mostly reflects how differently each rep's early rewards look. What to notice:

- **Early forced exploration:** UCB pulls each arm once before the bonus formula applies. That shows up as an initial burst for every arm.
- **Healthcare gradually dominates** — UCB concentrates mass on the arm with the highest (mean + bonus), which over enough time becomes Healthcare.
- **Loser arms do NOT decay to zero** — the $\sqrt{\log t / n_a}$ bonus is still large enough to occasionally win the argmax even thousands of steps in. UCB keeps sanity-checking its least-pulled arm forever, which is why the orange/blue curves hover around 3-5% rather than collapsing.
- **The curves are the smoothest of the three algorithms** precisely because UCB has almost no randomness: the only stochasticity is in the rewards themselves, not in the arm choice.
- **Error bars are narrower than epsilon-greedy** — different reps behave similarly because the deterministic policy makes essentially the same choice given similar histories.

UCB's top-arm share is typically similar to or slightly smaller than Thompson's on this problem; its advantage is that it **never fully abandons** any arm, which makes it robust to bad early draws. Thompson, by contrast, can abandon an arm permanently if it gets unlucky early.


In [ ]:
plot_alloc(alloc_results['ucb']['p_mean'], alloc_results['ucb']['p_std'],
           'UCB ($c=0.3$): $P(a_t = a)$ over time')


## Part 2, Plot 3 of 3 — Thompson Sampling allocation

Thompson sampling maintains a $\text{Beta}(\alpha_a, \beta_a)$ posterior per arm and pulls whichever arm's sampled draw is highest. Exploration is baked in: an arm with a wider posterior has a fatter tail and can win the argmax even when its mean is lower.

What to notice (and **not** notice — this is the key honest lesson of the plot):

- **Healthcare's allocation climbs gradually and wins eventually**, but on this problem with true rates $[0.10, 0.12, 0.08, 0.07, 0.11]$ — where the top three arms are within 0.02 of each other — Thompson does **not** instantly collapse onto Healthcare. After 2000 steps Healthcare's share is typically in the 25–40% range, not 90%+.
- **Unity (true 0.11) holds meaningful mass for the entire run.** Its posterior overlaps Healthcare's, so its sampled draws beat Healthcare's a non-trivial fraction of the time. This is not a bug — it's Thompson's rational response to residual uncertainty.
- **Economic (true 0.10) also holds meaningful mass**, for the same reason. Both arms are "plausibly the best" even after many pulls.
- **The worst arms (Climate 0.07, Education 0.08) decay slowly** — slower than you'd expect if gaps were large. Their allocation settles around 5-10% rather than zero because Thompson's prior ($\text{Beta}(1,1)$) keeps giving them nonzero posterior mass and occasional wins.
- **Error bars are widest on the top arms (Healthcare, Unity, Economic)** — different reps concentrate on different arms in different amounts, so the cross-rep std is large. This is Thompson's **exploration-as-uncertainty-sampling** in action: each rep plays a slightly different game depending on which early draws were lucky.

**The narrative moral.** When arm gaps are small (≤0.02 in a Bernoulli), no algorithm with 2000 pulls will collapse onto the true winner. Thompson is the best at *concentrating* what mass it can, but it still leaves 60-70% of pulls on the non-winning top arms because *they might actually be the winner*. This is exactly the regime Che & Namkoong target with RHO: small gaps, limited budget, high residual uncertainty.

**Compare to Lee & Green (2024)**, which had 11 arms and 2,168 respondents. Their winning arm ended up with a posterior probability of being best of only **0.19** — consistent with what you see here. Adaptive sampling identifies the *likely* winner but rarely produces a certified one when gaps are small.


In [ ]:
plot_alloc(alloc_results['thompson']['p_mean'], alloc_results['thompson']['p_std'],
           'Thompson Sampling: $P(a_t = a)$ over time')


## Part 2 Summary — Two views of the same run

Part 1 plotted the **estimates**: what does the algorithm *think* each arm's rate is, and how does that belief evolve? Part 2 plots the **actions**: at each moment, how likely is the algorithm to actually pull each arm?

The two plots answer different questions:

| View | Axis | Answers |
|------|------|---------|
| Part 1: $\hat\mu_a(t)$ | Arm's running mean estimate | *What does the algorithm know about arm $a$ at time $t$?* |
| Part 2: $P(a_t = a)$ | Probability of picking arm $a$ at step $t$ | *What does the algorithm do at time $t$?* |

**The honest lesson from Part 2.** When the top arms are close together (Healthcare 0.12, Unity 0.11, Economic 0.10 — all within 0.02), **none of the three algorithms** collapses onto Healthcare after 2000 pulls. Thompson gets closest, but still leaves substantial mass on Unity and Economic because their posteriors overlap Healthcare's. Epsilon-greedy sometimes commits to the wrong top arm and gets stuck there. UCB slowly concentrates but never fully abandons the losers.

This mirrors the real Lee & Green (2024) result: after 10 Thompson batches and 2,168 respondents, the posterior probability that the winning arm is actually best was only **0.19**. When ballot-wording effects are small, adaptive sampling gives you a *ranking with uncertainty*, not a certified winner.

**For HW4:** the scoring cares about *cumulative reward*, which means time spent on non-winning top-3 arms is not fully wasted — Unity (0.11) is almost as good as Healthcare (0.12). A strategy that concentrates hard on *any* top-3 arm will score almost as well as one that correctly identifies the true winner. The real loss is exploring Climate (0.07) and Education (0.08) beyond what's needed to rule them out.

Read Part 1 to see what the algorithms *know*. Read Part 2 to see what they *do*. The two don't always match — epsilon-greedy especially can know the right answer and still burn 10% of every step on pure random exploration forever.
